# Phase 1 — Data Collection & Merging

Loads all 7 raw sources, standardises country names and years, and merges into one `master_dataset.csv`.

Output: `../data/processed/master_dataset.csv`

In [ ]:
import pandas as pd
import numpy as np

RAW = '../data/raw/'
PROCESSED = '../data/processed/'

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

## 1. Country reference list

We define the European countries we want to keep across all datasets.
Using ISO3 codes as the common key.

In [ ]:
EUROPEAN_COUNTRIES = {
    'AUT': 'Austria',       'BEL': 'Belgium',       'BGR': 'Bulgaria',
    'HRV': 'Croatia',       'CYP': 'Cyprus',        'CZE': 'Czech Republic',
    'DNK': 'Denmark',       'EST': 'Estonia',       'FIN': 'Finland',
    'FRA': 'France',        'DEU': 'Germany',       'GRC': 'Greece',
    'HUN': 'Hungary',       'IRL': 'Ireland',       'ITA': 'Italy',
    'LVA': 'Latvia',        'LTU': 'Lithuania',     'LUX': 'Luxembourg',
    'MLT': 'Malta',         'NLD': 'Netherlands',   'NOR': 'Norway',
    'POL': 'Poland',        'PRT': 'Portugal',      'ROU': 'Romania',
    'SVK': 'Slovak Republic','SVN': 'Slovenia',     'ESP': 'Spain',
    'SWE': 'Sweden',        'CHE': 'Switzerland',   'GBR': 'United Kingdom',
    'ISL': 'Iceland',       'ALB': 'Albania',       'SRB': 'Serbia',
    'MKD': 'North Macedonia','MNE': 'Montenegro',   'BIH': 'Bosnia and Herzegovina'
}

# Region mapping for analysis
REGIONS = {
    'AUT': 'Western', 'BEL': 'Western', 'FRA': 'Western', 'DEU': 'Western',
    'IRL': 'Western', 'LUX': 'Western', 'NLD': 'Western', 'CHE': 'Western',
    'GBR': 'Western',
    'DNK': 'Northern', 'EST': 'Northern', 'FIN': 'Northern', 'ISL': 'Northern',
    'LVA': 'Northern', 'LTU': 'Northern', 'NOR': 'Northern', 'SWE': 'Northern',
    'CYP': 'Southern', 'GRC': 'Southern', 'ITA': 'Southern', 'MLT': 'Southern',
    'PRT': 'Southern', 'ESP': 'Southern',
    'ALB': 'Eastern', 'BIH': 'Eastern', 'BGR': 'Eastern', 'HRV': 'Eastern',
    'CZE': 'Eastern', 'HUN': 'Eastern', 'MKD': 'Eastern', 'MNE': 'Eastern',
    'POL': 'Eastern', 'ROU': 'Eastern', 'SRB': 'Eastern', 'SVK': 'Eastern',
    'SVN': 'Eastern'
}

print(f'Target: {len(EUROPEAN_COUNTRIES)} European countries')

## 2. WHO — Practising Nurses per 10,000

In [ ]:
who_raw = pd.read_csv(RAW + 'HLTHRES_189_EN.csv', skiprows=20, low_memory=False)
print('Raw shape:', who_raw.shape)
who_raw.head()

In [ ]:
who = who_raw[['COUNTRY', 'YEAR', 'VALUE']].copy()
who.columns = ['iso3', 'year', 'nurses_per_10k']
who = who.dropna(subset=['iso3', 'year', 'nurses_per_10k'])
who['iso3'] = who['iso3'].str.strip()
who['year'] = who['year'].astype(int)
who['nurses_per_10k'] = pd.to_numeric(who['nurses_per_10k'], errors='coerce')

# Keep only European countries and years 2000+
who = who[who['iso3'].isin(EUROPEAN_COUNTRIES.keys()) & (who['year'] >= 2000)]
who = who.drop_duplicates(subset=['iso3', 'year'])

print('Cleaned shape:', who.shape)
print('Countries:', sorted(who['iso3'].unique()))
print('Years:', sorted(who['year'].unique()))
who.head(10)

## 3. OECD — AMI & Stroke 30-day Mortality

In [ ]:
oecd_raw = pd.read_csv(RAW + 'OECD.ELS.csv', low_memory=False)
print('Raw shape:', oecd_raw.shape)

# MORTAMII = AMI 30-day in-hospital mortality
# MORTISTI = Ischaemic stroke 30-day in-hospital mortality
oecd_outcomes = oecd_raw[oecd_raw['MEASURE'].isin(['MORTAMII', 'MORTISTI'])].copy()

# Keep total (not broken down by age/sex)
if 'SEX' in oecd_outcomes.columns:
    oecd_outcomes = oecd_outcomes[oecd_outcomes['SEX'].isin(['_T', 'T', '_Z', 'Total', 'SEX'])]
if 'AGE' in oecd_outcomes.columns:
    oecd_outcomes = oecd_outcomes[oecd_outcomes['AGE'].isin(['_T', 'T', '_Z', 'Total', 'Y_GE15'])]

print('After filter:', oecd_outcomes.shape)
print('Measures:', oecd_outcomes['MEASURE'].unique())

In [ ]:
# Pivot to wide format: one row per country-year
oecd_outcomes['iso3'] = oecd_outcomes['REF_AREA']
oecd_outcomes['year'] = oecd_outcomes['TIME_PERIOD'].astype(int)
oecd_outcomes['value'] = pd.to_numeric(oecd_outcomes['OBS_VALUE'], errors='coerce')

oecd_wide = oecd_outcomes.groupby(['iso3', 'year', 'MEASURE'])['value'].mean().unstack('MEASURE').reset_index()
oecd_wide.columns.name = None

# Rename to our variable names
rename = {'MORTAMII': 'mortality_ami_30d', 'MORTISTI': 'mortality_stroke_30d'}
oecd_wide = oecd_wide.rename(columns=rename)

# Keep only European countries
oecd_wide = oecd_wide[oecd_wide['iso3'].isin(EUROPEAN_COUNTRIES.keys())]

print('OECD outcomes shape:', oecd_wide.shape)
print('Countries:', sorted(oecd_wide['iso3'].unique()))
oecd_wide.head(10)

## 4. OECD — Average Length of Stay

In [ ]:
los_raw = pd.read_csv(RAW + 'OECD_ELS_HD_DSD_HEALTH_PROC.csv', low_memory=False)
print('Raw shape:', los_raw.shape)

# Filter to STAY measure, inpatient, all causes total
los = los_raw[
    (los_raw['MEASURE'] == 'STAY') &
    (los_raw['CARE_TYPE'].isin(['_T', 'T', 'Total'])) &
    (los_raw['FUNCTION'].isin(['_T', 'T', 'Total']))
].copy()

print('After filter:', los.shape)
los[['REF_AREA', 'Reference area', 'TIME_PERIOD', 'OBS_VALUE']].head(10)

In [ ]:
los['iso3'] = los['REF_AREA']
los['year'] = los['TIME_PERIOD'].astype(int)
los['avg_length_of_stay'] = pd.to_numeric(los['OBS_VALUE'], errors='coerce')

los = los[['iso3', 'year', 'avg_length_of_stay']]
los = los[los['iso3'].isin(EUROPEAN_COUNTRIES.keys()) & (los['year'] >= 2000)]
los = los.dropna(subset=['avg_length_of_stay'])
los = los.groupby(['iso3', 'year'])['avg_length_of_stay'].mean().reset_index()

print('LOS shape:', los.shape)
print('Countries:', sorted(los['iso3'].unique()))
los.head(10)

## 5. World Bank — Beds, Health Expenditure, GDP

In [ ]:
def load_worldbank(filepath, varname):
    df = pd.read_csv(filepath, skiprows=4, low_memory=False)
    # Keep only country code + year columns
    year_cols = [c for c in df.columns if c.isdigit()]
    df = df[['Country Code'] + year_cols].copy()
    df = df[df['Country Code'].isin(EUROPEAN_COUNTRIES.keys())]
    # Melt to long format
    df = df.melt(id_vars='Country Code', var_name='year', value_name=varname)
    df.columns = ['iso3', 'year', varname]
    df['year'] = df['year'].astype(int)
    df[varname] = pd.to_numeric(df[varname], errors='coerce')
    df = df[df['year'] >= 2000].dropna(subset=[varname])
    return df

beds = load_worldbank(RAW + 'API_SH.MED.BEDS.ZS_DS2_en_csv_v2_115646.csv', 'beds_per_1000')
health_exp = load_worldbank(RAW + 'API_SH.XPD.CHEX.GD.ZS_DS2_en_csv_v2_115482.csv', 'health_exp_pct_gdp')
gdp = load_worldbank(RAW + 'API_NY.GDP.PCAP.CD_DS2_en_csv_v2_121663.csv', 'gdp_per_capita')

print('Beds:', beds.shape)
print('Health Exp:', health_exp.shape)
print('GDP:', gdp.shape)

## 6. Eurostat — Cross-check Nurses (practising, not licensed)

In [ ]:
estat_raw = pd.read_csv(RAW + 'eurostat_nurses.csv', low_memory=False)
print('Columns:', estat_raw.columns.tolist())
print('wstatus unique:', estat_raw['wstatus'].unique())
print('med_spec unique:', estat_raw['med_spec'].unique()[:10])
print('unit unique:', estat_raw['unit'].unique())

In [ ]:
# Keep practising nurses (not licensed), per 100k inhabitants
estat = estat_raw[
    (estat_raw['wstatus'] == 'PRF') &
    (estat_raw['med_spec'] == 'NURSE') &
    (estat_raw['unit'] == 'HAB_P')
].copy()

print('After filter:', estat.shape)
estat[['geo', 'TIME_PERIOD', 'OBS_VALUE']].head(10)

In [ ]:
# Note: Eurostat uses 2-letter ISO codes - map to ISO3
iso2_to_iso3 = {
    'AT':'AUT','BE':'BEL','BG':'BGR','HR':'HRV','CY':'CYP','CZ':'CZE',
    'DK':'DNK','EE':'EST','FI':'FIN','FR':'FRA','DE':'DEU','GR':'GRC',
    'HU':'HUN','IE':'IRL','IT':'ITA','LV':'LVA','LT':'LTU','LU':'LUX',
    'MT':'MLT','NL':'NLD','NO':'NOR','PL':'POL','PT':'PRT','RO':'ROU',
    'SK':'SVK','SI':'SVN','ES':'ESP','SE':'SWE','CH':'CHE','GB':'GBR',
    'IS':'ISL','AL':'ALB','RS':'SRB','MK':'MKD','ME':'MNE','BA':'BIH'
}

estat['iso3'] = estat['geo'].map(iso2_to_iso3)
estat['year'] = estat['TIME_PERIOD'].astype(int)
estat['nurses_eurostat_per_100k'] = pd.to_numeric(estat['OBS_VALUE'], errors='coerce')

estat = estat[['iso3', 'year', 'nurses_eurostat_per_100k']]
estat = estat.dropna(subset=['iso3', 'nurses_eurostat_per_100k'])
estat = estat[estat['year'] >= 2000]

print('Eurostat nurses shape:', estat.shape)
print('Countries:', sorted(estat['iso3'].unique()))

## 7. Merge all datasets

WHO nurses is the primary staffing source. We merge everything on `iso3` + `year`.

In [ ]:
# Build base: all country-year combos from 2000-2023
from itertools import product

all_iso3 = list(EUROPEAN_COUNTRIES.keys())
all_years = list(range(2000, 2024))
base = pd.DataFrame(list(product(all_iso3, all_years)), columns=['iso3', 'year'])
base['country'] = base['iso3'].map(EUROPEAN_COUNTRIES)
base['region'] = base['iso3'].map(REGIONS)
base['covid_period'] = base['year'].apply(
    lambda y: 'pre' if y <= 2019 else ('during' if y <= 2021 else 'post')
)

print('Base panel:', base.shape)

# Merge one by one
master = base.copy()
master = master.merge(who, on=['iso3', 'year'], how='left')
master = master.merge(oecd_wide, on=['iso3', 'year'], how='left')
master = master.merge(los, on=['iso3', 'year'], how='left')
master = master.merge(beds, on=['iso3', 'year'], how='left')
master = master.merge(health_exp, on=['iso3', 'year'], how='left')
master = master.merge(gdp, on=['iso3', 'year'], how='left')
master = master.merge(estat, on=['iso3', 'year'], how='left')

print('Master shape:', master.shape)
master.head(10)

In [ ]:
# Engineer nurse change rate (YoY % change)
master = master.sort_values(['iso3', 'year'])
master['nurse_change_rate'] = master.groupby('iso3')['nurses_per_10k'].pct_change() * 100

print('Final columns:', master.columns.tolist())
print('\nShape:', master.shape)

In [ ]:
# Missing data summary
missing = (master.isnull().sum() / len(master) * 100).round(1)
print('Missing % per column:')
print(missing.to_string())

In [ ]:
# Save
master.to_csv(PROCESSED + 'master_dataset.csv', index=False)
print(f'Saved master_dataset.csv — {master.shape[0]} rows x {master.shape[1]} columns')
master.describe().round(2)